<a href="https://colab.research.google.com/github/GLzanetti/Projetos-de-Machine-Learning/blob/main/Aula3_MachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 3 de Machine Learning - Modelo de Árvores de Decisão/Floresta Randomica

## Regressão de Preços de Imóveis com Scikit-Learn

Este projeto consiste em uma aplicação prática de **Aprendizado de Máquina Supervisionado** voltada para a tarefa de **Regressão**. Utilizando a biblioteca Scikit-Learn, construímos e comparamos modelos preditivos com diferentes níveis de complexidade (Regressão Linear, Árvore de Decisão e Floresta Aleatória) para estimar o valor de venda de imóveis na cidade de São Paulo. Através dessa análise comparativa, exploramos como diferentes arquiteturas de algoritmos lidam com a complexidade e a variabilidade dos dados reais do mercado imobiliário.

### 📋 Objetivo
O objetivo deste projeto é prever o preço de venda de imóveis na cidade de São Paulo com base em suas características estruturais e de localização (como tamanho, número de quartos, banheiros, vagas de garagem, etc.).

### 🛠️ Detalhes da Implementação
- **Dataset do Kaggle**: Utiliza a base de dados `argonalyst/sao-paulo-real-estate-sale-rent-april-2019` contendo informações detalhadas sobre imóveis em São Paulo.
- **Modelos Avaliados**:
  1. **Regressão Linear**: Modelo de linha de base para relações lineares.
  2. **Árvore de Decisão (Decision Tree)**: Modelo não-linear baseado em regras de decisão estruturadas.
  3. **Floresta Aleatória (Random Forest)**: Modelo de ensemble que combina múltiplas árvores de decisão para melhorar a robustez e precisão.
- **Métricas de Performance Calculadas**:
  - **MAE (Erro Absoluto Médio)**: Média das diferenças absolutas entre os valores previstos e reais.
  - **MAPE (Erro Percentual Absoluto Médio)**: Média do percentual de erro em relação aos valores reais.
  - **R2 (Coeficiente de Determinação)**: Proporção da variância dos preços de imóveis explicada pelo modelo.
  - **Assertividade**: Métrica customizada definida como `max(0, 100 * (1 - MAPE))`.

In [ ]:
#Instalação da biblioteca do Kaggle

!pip install -q kagglehub

In [ ]:
#Import das libs

import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

print("1) Baixando dados do Kaggle")

pasta = kagglehub.dataset_download("argonalyst/sao-paulo-real-estate-sale-rent-april-2019")
df = pd.read_csv(pasta + "/sao-paulo-properties-april-2019.csv")

print("Tamanho do dataset: ", len(df))
#display(df) #Mostra a tabela dos dados do dataset

print("\nPreparando os dados")
colunas = ['Condo', 'Size', 'Rooms', 'Toilets', 'Suites', 'Parking']
df = df[df['Negotiation Type'] == 'sale'].copy()

print("Novo tamanho do dataset: ", len(df))

X = df[colunas] #Variaves com letras maiusculas em python representam constantes, esse X representa o x da equacao da reta
y = df['Price'] #Esse y representa o valor de previsao

print("\nSeparando dados de treino e teste")

xTreino, xTeste, yTreino, yTeste = train_test_split(X, y, test_size=0.3, random_state=42)

modelos = {"linear": LinearRegression(),
           "arvore": DecisionTreeRegressor(max_depth=None, min_samples_leaf=1, random_state=42), #Uderfitting = (max_depth=1, min_samples_leaf=200, random_state=42) overfitting = (max_depth=None, min_samples_leaf=1, random_state=42)
           "floresta": RandomForestRegressor(max_depth=None, n_estimators=300, min_samples_leaf=1, max_features=1.0, random_state=42) #Underfitting (max_depth=2, n_estimators=20, min_samples_leaf=100, max_features=0.3, random_state=42) overfitting = (max_depth=None, n_estimators=300, min_samples_leaf=1, max_features=1, random_state=42)
}

previsores = {}
resultados = {}

print("\n4) Treinando os modelos")

for nome, modelo in modelos.items():
  print("- Treinando modelo de: ", nome,"...")
  modelo.fit(xTreino, yTreino)
  previsores[nome] = modelo.predict(xTeste)

  mae = mean_absolute_error(yTeste, previsores[nome])
  mape = mean_absolute_percentage_error(yTeste, previsores[nome])
  r2 = r2_score(yTeste, previsores[nome])

  assertividade = max(0, 100*(1-mape))
  resultados[nome] = [mae, r2, assertividade]

print("\n5) Resultados dos modelos")
print("===========================")

for nome, (mae, r2, assertividade) in resultados.items():
  print(f"{nome:8}")
  print(f"MAE: {mae:.2f}")
  print(f"R2: {r2:.2f}")
  print(f"Assertividade: {assertividade:.2f}%\n")
  print("===============================")

print("\n6) Testando com um dado novo")
apartamento = xTeste.iloc[[0]]
precoReal = yTeste.iloc[0]
print("Apartamento: ", apartamento)
print("Preço real: ", precoReal)

for nome, modelo in modelos.items():
  precoPrevisto = modelo.predict(apartamento)[0]
  erro = abs(precoPrevisto - precoReal) / precoReal * 100
  acertou = "Sim" if erro <= 20 else "Não"
  print(f"{nome:8} -> Preco previsto: {precoPrevisto:,.0f}")
  print(f"Erro {erro:.2f}% Acertou? {acertou}")



1) Baixando dados do Kaggle
Using Colab cache for faster access to the 'sao-paulo-real-estate-sale-rent-april-2019' dataset.
Tamanho do dataset:  13640

Preparando os dados
Novo tamanho do dataset:  6412

Separando dados de treino e teste

4) Treinando os modelos
- Treinando modelo de:  linear ...
- Treinando modelo de:  arvore ...
- Treinando modelo de:  floresta ...

5) Resultados dos modelos
linear  
MAE: 196953.15
R2: 0.68
Assertividade: 63.21%

arvore  
MAE: 186754.05
R2: 0.67
Assertividade: 70.35%

floresta
MAE: 150964.62
R2: 0.79
Assertividade: 75.40%


6) Testando com um dado novo
Apartamento:        Condo  Size  Rooms  Toilets  Suites  Parking
5722    430    47      1        2       1        1
Preço real:  329000
linear   -> Preco previsto: 460,805
Erro 40.06% Acertou? Não
arvore   -> Preco previsto: 249,500
Erro 24.16% Acertou? Não
floresta -> Preco previsto: 269,871
Erro 17.97% Acertou? Sim
